# 1. Data Understanding

In [ ]:
import os
import pandas as pd
from IPython.display import display

# ==========================================
# CONFIGURATION & GLOBAL VARIABLES
# ==========================================
DATASET_PATH = '/kaggle/input/datasets/tranphamthanhtruc/theme-3-dataset'

# Dictionary to store all successfully loaded DataFrames.
# Key = File name (without .csv extension), Value = Corresponding DataFrame.
dataframes_dict = {}

def check_data_understanding(dataset_path):
    """
    Recursively scans the directory for CSV files, handles encoding errors 
    during ingestion, and performs initial Exploratory Data Analysis (EDA).
    
    Args:
        dataset_path (str): The root directory path containing the dataset.
        
    Action:
        - Prints dataset dimensions (shape).
        - Checks for and reports missing values (NaNs).
        - Displays a preview of the first 5 rows for quick inspection.
    """
    print(f"Starting to scan directory: {dataset_path}\n" + "="*50)
    
    # Recursively traverse through all subdirectories and files
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.endswith('.csv'):
                # Construct the absolute file path and extract the base file name
                file_path = os.path.join(root, file)
                file_name = os.path.splitext(file)[0] 
                
                print(f"\n📁 Opening file: {file} (from folder: {os.path.basename(root)})")
                
                # ---------------------------------------------------------
                # STEP 1: ENCODING ERROR HANDLING
                # ---------------------------------------------------------
                # Industrial/SCADA system exports often use UTF-16 instead of 
                # standard UTF-8. We attempt UTF-8 first, then fallback to UTF-16.
                try:
                    df = pd.read_csv(file_path, encoding='utf-8')
                except UnicodeDecodeError:
                    print(f"  ⚠️ UTF-8 decoding failed. Trying UTF-16 encoding...")
                    try:
                        df = pd.read_csv(file_path, encoding='utf-16')
                    except Exception as e:
                        print(f"  ❌ Error reading file {file} with UTF-16: {e}")
                        continue # Skip to the next file if fallback also fails
                except Exception as e:
                    print(f"  ❌ Unexpected error reading file {file}: {e}")
                    continue

                # If successfully read, store it in the global dictionary for later use
                dataframes_dict[file_name] = df
                
                # ---------------------------------------------------------
                # STEP 2: INITIAL DATA PROFILING (EDA)
                # ---------------------------------------------------------
                
                # 2.1. Check data dimensions (Rows x Columns)
                print(f"  - Shape: {df.shape[0]} rows, {df.shape[1]} columns")
                
                # 2.2. Identify Missing Values (NaNs)
                missing_data = df.isnull().sum()
                cols_with_missing = missing_data[missing_data > 0]
                
                if not cols_with_missing.empty:
                    print(f"  - Missing Values found in {len(cols_with_missing)} columns.")
                else:
                    print("  - No Missing Values found.")
                    
                # 2.3. Statistical Distribution Review
                print("  - Statistical Summary (describe):")
                numeric_cols = df.select_dtypes(include='number').columns
                if not numeric_cols.empty:
                    # Hạn chế số lượng cột hiển thị trong distribution để không quá rối
                    display(df[numeric_cols].describe().T.head(10)) 
                else:
                    print("    No numeric columns found.")
                    
                # 2.4. Visual Data Preview (Highly recommended for Jupyter/Kaggle environments)
                print("  - Data Preview (First 5 rows):")
                display(df.head(5)) 
                print("-" * 50)

# ==========================================
# EXECUTION BLOCK
# ==========================================
# Safely check if the directory exists before executing to prevent FileNotFoundError
if os.path.exists(DATASET_PATH):
    check_data_understanding(DATASET_PATH)
else:
    print(f"⚠️ Path not found: {DATASET_PATH}")


## Yeast Type && Missing Value

In [ ]:
import pandas as pd
import os

# ==========================================
# CONFIGURATION & GLOBAL VARIABLES
# ==========================================
DATASET_PATH = '/kaggle/input/datasets/tranphamthanhtruc/theme-3-dataset/Theme3'
OUTPUT_PATH = '/kaggle/working/Theme3_Merged_Quality_Data_FIXED.csv'

def merge_quality_sensor_data(dataset_path, output_path):
    """
    Scans the dataset directory for wide-format SCADA sensor files, standardizes 
    the target labels based on filenames, and merges them into a single master dataset.
    
    This function explicitly filters out non-sensor directories and strictly enforces 
    schema requirements to prevent data contamination. It also handles overlapping 
    records by applying a strict deduplication strategy.
    
    Args:
        dataset_path (str): Root directory containing the raw SCADA CSV files.
        output_path (str): Destination path for the final merged CSV.
    """
    print("🔍 1. SCANNING AND MERGING ALL AVAILABLE WIDE-FORMAT FILES...\n" + "="*50)

    # Initialize a list to hold all valid DataFrames containing SP/PV sensor data
    frames_to_merge = []

    # ---------------------------------------------------------
    # STEP 1: DIRECTORY TRAVERSAL & FILE FILTERING
    # ---------------------------------------------------------
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                
                # Context Isolation: Skip 'Downtime' and 'Infinity' directories 
                # as they possess fundamentally different schemas and granularities.
                if 'Downtime' in root or 'Infinity' in root:
                    continue
                    
                try:
                    # Encoding Fallback: Industrial SCADA systems frequently export 
                    # in UTF-16. Attempt standard UTF-8 first, fallback if necessary.
                    try:
                        df_temp = pd.read_csv(file_path, encoding='utf-8')
                    except UnicodeDecodeError:
                        df_temp = pd.read_csv(file_path, encoding='utf-16')
                    
                    # ---------------------------------------------------------
                    # STEP 2: SCHEMA ENFORCEMENT & TARGET EXTRACTION
                    # ---------------------------------------------------------
                    # Strict Schema Validation: Only accept "wide-format" files 
                    # (>= 30 columns) to guarantee all necessary sensors are present, 
                    # actively filtering out narrow summary logs (7-11 columns).
                    if df_temp.shape[1] >= 30:
                        
                        # Target Variable Labeling: Infer the quality class logically 
                        # from the standardized file naming conventions.
                        file_lower = file.lower()
                        if 'good' in file_lower:
                            df_temp['Quality_Label'] = 'Good'
                        elif 'low' in file_lower:
                            df_temp['Quality_Label'] = 'Low_Bad'
                        elif 'high' in file_lower:
                            df_temp['Quality_Label'] = 'High_Bad'
                        else:
                            # If the file is a generic 'mixed' or 'process_all' file, 
                            # we process it if it already contains a mapped 'Class' column
                            if 'Class' in df_temp.columns:
                                df_temp['Quality_Label'] = df_temp['Class'].astype(str)
                            else:
                                continue
                                
                        frames_to_merge.append(df_temp)
                        print(f"  ✅ Picked up valid Wide file: {file} | Shape: {df_temp.shape}")
                        
                except Exception as e:
                    print(f"  ❌ Error reading {file}: {e}")

    # ---------------------------------------------------------
    # STEP 3: CONSOLIDATION & DEDUPLICATION
    # ---------------------------------------------------------
    if not frames_to_merge:
        print("\n❌ CRITICAL ERROR: Could not find any valid files.")
    else:
        # Concatenate all validated dataframes into a single master dataframe
        df_merged = pd.concat(frames_to_merge, ignore_index=True)
        print(f"\n🚀 Raw merged shape: {df_merged.shape}")
        
        # Data Integrity Check: Remove overlapping records. Overlaps often occur 
        # when 'mixed' files contain the same batch data as specific quality files.
        # We use a composite key (Batch ID + Machine Part + Timestamp) to ensure uniqueness.
        df_merged = df_merged.drop_duplicates(subset=['VYP batch', 'Part', 'Set Time'])
        print(f"🧹 Shape after removing duplicates: {df_merged.shape}")
        
        # Export the clean, ML-ready dataset
        df_merged.to_csv(output_path, index=False)
        print(f"💾 Saved fixed master file to: {output_path}")

    print("\n🎉 DATA IS NOW CORRECTLY MERGED WITH NO DUPLICATES.")

# ==========================================
# EXECUTION
# ==========================================
if os.path.exists(DATASET_PATH):
    merge_quality_sensor_data(DATASET_PATH, OUTPUT_PATH)
else:
    print(f"⚠️ Path not found: {DATASET_PATH}")


## Downtime

In [ ]:
import pandas as pd
import os
from IPython.display import display

# ==========================================
# CONFIGURATION & GLOBAL VARIABLES
# ==========================================
DATASET_PATH = '/kaggle/input/datasets/tranphamthanhtruc/theme-3-dataset/Theme3/Downtime'
OUTPUT_PATH = '/kaggle/working/Theme3_Cleaned_Downtime.csv'

def clean_downtime_logs(dataset_path, output_path):
    """
    Locates the specific Downtime event log, standardizes data types, 
    and unifies fragmented date/time strings into a single, chronologically 
    sorted datetime object.

    This preprocessing step is critical for Time-Series tasks, as these 
    timestamps will act as the anchor points for generating 'Warning Windows' 
    in the predictive maintenance pipeline.

    Args:
        dataset_path (str): Directory containing the downtime logs.
        output_path (str): Destination path for the cleaned CSV.
    """
    downtime_file_path = None

    # ---------------------------------------------------------
    # STEP 1: FILE DISCOVERY
    # ---------------------------------------------------------
    # Scan the directory specifically for the 'Yeast Prep DT' file, 
    # isolating it from other irrelevant logs in the folder.
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if 'Yeast Prep DT' in file and file.endswith('.csv'):
                downtime_file_path = os.path.join(root, file)
                break

    if downtime_file_path:
        # 1. LOAD THE DATA
        df_downtime = pd.read_csv(downtime_file_path)
        print("🔧 FIXING DOWNTIME DATA FORMATS...\n" + "="*50)
        
        # ---------------------------------------------------------
        # STEP 2: DATA TYPE CASTING (NUMERIC)
        # ---------------------------------------------------------
        # Manual industrial logs often contain accidental string characters (e.g., "10 mins" 
        # instead of just "10"). We force numeric conversion, coercing errors to NaN 
        # so they don't break downstream mathematical operations.
        if 'Total Time Mins' in df_downtime.columns:
            df_downtime['Total Time Mins'] = pd.to_numeric(df_downtime['Total Time Mins'], errors='coerce')
            print("✅ Fixed 'Total Time Mins': Successfully converted to numeric duration.")

        # ---------------------------------------------------------
        # STEP 3: TEMPORAL UNIFICATION & SORTING
        # ---------------------------------------------------------
        if 'Production Date' in df_downtime.columns and 'Time' in df_downtime.columns:
            # Combine fragmented Date and Time strings into a single timestamp string
            combined_dt_str = df_downtime['Production Date'].astype(str) + ' ' + df_downtime['Time'].astype(str)
            
            # CRITICAL FIX: Pandas defaults to the US date format (MM/DD/YYYY). 
            # Industrial logs in many regions use DD/MM/YYYY. Setting dayfirst=True 
            # prevents catastrophic misalignment of months and days.
            df_downtime['Start_Datetime'] = pd.to_datetime(combined_dt_str, dayfirst=True, errors='coerce')
            
            # Sort chronologically. This is strictly required because future time-window 
            # mapping (e.g., looking back 30 minutes) relies on ordered temporal data.
            df_downtime = df_downtime.sort_values(by='Start_Datetime').reset_index(drop=True)
            print("✅ Created new unified timestamp: 'Start_Datetime' (Sorted chronologically).")
            
            # ---------------------------------------------------------
            # STEP 4: VALIDATION & PREVIEW
            # ---------------------------------------------------------
            print("\n📊 CLEANED TIME COLUMNS PREVIEW:")
            display(df_downtime[['Production Date', 'Time', 'Start_Datetime', 'Total Time Mins']].head())
            
            # Data Integrity Check: Ensure no timestamps were lost during the string conversion
            failed_conversions = df_downtime['Start_Datetime'].isna().sum()
            if failed_conversions > 0:
                print(f"⚠️ Warning: {failed_conversions} rows failed datetime conversion.")
            else:
                print("🎉 Success: 100% of rows converted to datetime perfectly.")
        
        # ---------------------------------------------------------
        # STEP 5: EXPORT ML-READY LOGS
        # ---------------------------------------------------------
        df_downtime.to_csv(output_path, index=False)
        print(f"\n💾 Saved the properly formatted file to: {output_path}")

    else:
        print("❌ Could not locate the 'Yeast Prep DT' file. Check your dataset folder paths.")

# ==========================================
# EXECUTION
# ==========================================
if os.path.exists(DATASET_PATH):
    clean_downtime_logs(DATASET_PATH, OUTPUT_PATH)
else:
    print(f"⚠️ Path not found: {DATASET_PATH}")

## Sensor Trends Missing Values

In [ ]:
"""
Sensor Trend Data Pre-processing Pipeline

This script automatically scans, ingests, and cleans industrial continuous 
sensor data (Trend files). It addresses common SCADA data issues including:
1. Encoding inconsistencies (handling UTF-16 exports).
2. Temporal alignment (parsing DD/MM/YYYY formats and sorting chronologically).
3. Time-Series Missing Value Imputation (applying Forward-Fill to maintain 
   physical continuity and strictly avoid future data leakage).
"""

import os
import pandas as pd
from IPython.display import display

# Define the root directory for the dataset
DATASET_PATH = '/kaggle/input/datasets/tranphamthanhtruc/theme-3-dataset/Theme3' 

# Dictionary to hold the cleaned DataFrames in memory
sensor_dfs = {}

print("🔍 SCANNING FOR SENSOR TREND FILES...\n" + "="*50)

# Traverse the directory tree to find all relevant sensor files
for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        # ---------------------------------------------------------
        # FILE FILTERING
        # ---------------------------------------------------------
        # Looking for files that contain 'Trend' in their name
        if 'Trend' in file and file.endswith('.csv'):
            file_path = os.path.join(root, file)
            file_name = os.path.splitext(file)[0]
            
            print(f"\n📂 Processing: {file}")
            try:
                # ---------------------------------------------------------
                # 1. READ WITH UTF-16 ENCODING
                # ---------------------------------------------------------
                # Industrial systems (like Rockwell/Allen-Bradley) often export 
                # trend data in UTF-16. This forces pandas to read it correctly.
                df = pd.read_csv(file_path, encoding='utf-16')
                print(f"  ✅ Loaded successfully (UTF-16). Shape: {df.shape[0]} rows, {df.shape[1]} columns")
                
                # ---------------------------------------------------------
                # 2. FIX TIME COLUMN & CHRONOLOGICAL SORTING
                # ---------------------------------------------------------
                # Dynamically locate the time column, as naming conventions 
                # might slightly differ across various machine exports.
                time_col = None
                for col in df.columns:
                    if 'time' in col.lower():
                        time_col = col
                        break
                        
                if time_col:
                    print(f"  ⏳ Converting '{time_col}' to Datetime...")
                    # Using dayfirst=True to properly parse DD/MM/YYYY formats, 
                    # preventing catastrophic month/day swapping errors.
                    df[time_col] = pd.to_datetime(df[time_col], dayfirst=True, errors='coerce')
                    
                    # Sorting chronologically is an absolute prerequisite for 
                    # any time-series task (like rolling averages or imputation).
                    df = df.sort_values(by=time_col).reset_index(drop=True)
                    
                    failed_times = df[time_col].isna().sum()
                    if failed_times > 0:
                        print(f"  ⚠️ Warning: {failed_times} rows failed time conversion.")
                else:
                    print("  ⚠️ No Time column found in this file!")

                # ---------------------------------------------------------
                # 3. HANDLE MISSING SENSOR VALUES (TIME-SERIES IMPUTATION)
                # ---------------------------------------------------------
                missing_data = df.isnull().sum()
                cols_with_missing = missing_data[missing_data > 0]
                
                if not cols_with_missing.empty:
                    print(f"  ⚠️ Missing values detected in {len(cols_with_missing)} sensor columns.")
                    print("  🔧 Applying Forward Fill (ffill) for time-series continuity...")
                    
                    # Forward fill (ffill) respects the "arrow of time". It uses the 
                    # last known good sensor reading to fill the gap, ensuring Zero Data Leakage.
                    df = df.ffill() 
                    
                    # Backward fill (bfill) is used purely as a safety net for any NaNs 
                    # located at the very first row of the dataset.
                    if df.isnull().sum().sum() > 0:
                        df = df.bfill()
                        
                    print("  ✅ All missing values handled.")
                else:
                    print("  ✅ No missing sensor values detected.")

                # Save the cleaned dataframe to our dictionary for in-memory use
                sensor_dfs[file_name] = df
                
                # ---------------------------------------------------------
                # 4. EXPORT & PREVIEW
                # ---------------------------------------------------------
                # Replace spaces and hyphens with underscores to create safe, 
                # machine-readable filenames for the output directory.
                safe_name = file_name.replace(" ", "_").replace("-", "_")
                clean_path = f'/kaggle/working/Cleaned_{safe_name}.csv'
                df.to_csv(clean_path, index=False)
                print(f"  💾 Saved cleaned file to: {clean_path}")
                
                # PREVIEW: Display the first 3 rows to verify data integrity
                display(df.head(3))
                
            except Exception as e:
                # Catch and log any unexpected errors (e.g., corrupted files) 
                # without crashing the entire pipeline loop.
                print(f"  ❌ Failed to process {file}. Error: {e}")

print("\n🎉 FINISHED PROCESSING SENSOR DATA.")

## Merge Downtime

In [ ]:
"""
Downtime Event & Sensor Data Integration Pipeline

ML UPGRADE:
- Tính toán Sliding Window Features (mean, std, min, max, delta) TRƯỚC KHI MERGE.
- Tách riêng Machine Context (TFE, FFTE, Extract) để tránh nhiễu do gộp chung.
"""

import pandas as pd
import numpy as np
from IPython.display import display
import glob
import os
import re

print("🔗 MERGING DOWNTIME EVENTS WITH SENSOR DATA...\n" + "="*50)

dt_path = '/kaggle/working/Theme3_Cleaned_Downtime.csv'

try:
    df_dt = pd.read_csv(dt_path)
    df_dt['Start_Datetime'] = pd.to_datetime(df_dt['Start_Datetime'])
    df_dt = df_dt.drop_duplicates().sort_values(by='Start_Datetime')
    
    # Try to find a reason/category column for failure type
    reason_col = next((col for col in ['Reason', 'Category', 'Fault', 'Description', 'Equipment', 'Downtime Reason', 'Reason/Fault'] if col in df_dt.columns), None)
    
    print("🔄 Loading ALL cleaned sensor files to provide complete system context...")
    sensor_files = glob.glob('/kaggle/working/Cleaned_*.csv')
    sensor_files = [f for f in sensor_files if 'Downtime' not in f and 'Quality' not in f]
    
    processed_dfs = []
    
    for f in sensor_files:
        df_temp = pd.read_csv(f)
        
        # 1. ĐỊNH DANH MACHINE (Khắc phục lỗi rò rỉ tên nhãn - Dirty Label)
        fname = os.path.basename(f).upper()
        if 'TFE' in fname and 'FFTE' not in fname:
            machine_name = 'TFE'
        elif 'FFTE' in fname:
            machine_name = 'FFTE'
        elif 'EXTRACT' in fname:
            machine_name = 'EXTRACT'
        elif 'EVAPORATOR' in fname:
            machine_name = 'EVAPORATOR'
        else:
            machine_name = os.path.basename(f).replace('Cleaned_', '').replace('.csv', '')
            # Clean up dates, 'Trends', and trailing underscores from generic names
            machine_name = re.sub(r'(?i)_trends.*', '', machine_name)  # Remove '_Trends' and anything after it
            machine_name = re.sub(r'_\d+.*', '', machine_name)         # Remove any remaining date patterns
            machine_name = machine_name.upper().strip('_')
            
        df_temp['Machine_Type'] = machine_name
        
        time_cols = [c for c in df_temp.columns if 'time' in c.lower()]
        if time_cols:
            df_temp.rename(columns={time_cols[0]: 'Time'}, inplace=True)
            df_temp['Time'] = pd.to_datetime(df_temp['Time'])
            df_temp = df_temp.sort_values(by='Time').drop_duplicates().reset_index(drop=True)
            
        # ==========================================
        # 2. SLIDING WINDOW FEATURE ENGINEERING (PER MACHINE)
        # ==========================================
        print(f"  -> Engineering Window Features for Machine: {machine_name} | Rows: {len(df_temp)}")
        numeric_cols = df_temp.select_dtypes(include='number').columns
        
        # Chọn top 30 sensor để tối ưu RAM. Trong thực tế có thể apply toàn bộ.
        working_sensors = numeric_cols[:30] 
        window = 10 
        
        for col in working_sensors:
            df_temp[f'{col}_mean'] = df_temp[col].rolling(window=window, min_periods=1).mean()
            df_temp[f'{col}_std']  = df_temp[col].rolling(window=window, min_periods=1).std().fillna(0)
            df_temp[f'{col}_max']  = df_temp[col].rolling(window=window, min_periods=1).max()
            df_temp[f'{col}_min']  = df_temp[col].rolling(window=window, min_periods=1).min()
            df_temp[f'{col}_delta']= df_temp[col] - df_temp[col].shift(window).fillna(method='bfill')
            
        # ==========================================
        # 3. TIME-WINDOW LABELING (PER MACHINE WITH REASON)
        # ==========================================
        df_temp['Anomaly_Warning'] = 0
        df_temp['Failure_Type'] = 'None'
        # WARNING WINDOW FIX: Thu hẹp window xuống 15 phút để tăng Recall, giảm nhiễu
        WARNING_MINUTES = 15
        LEAD_TIME_MINUTES = 5
        warning_window = pd.Timedelta(minutes=WARNING_MINUTES)
        lead_window = pd.Timedelta(minutes=LEAD_TIME_MINUTES)
        
        match_count = 0
        for idx, row in df_dt.iterrows():
            dt_event = row['Start_Datetime']
            if pd.isna(dt_event): continue
            
            reason_val = 'Unknown'
            if reason_col:
                reason_val = str(row[reason_col])
                
            start_warning = dt_event - warning_window
            end_warning = dt_event - lead_window
            
            mask = (df_temp['Time'] >= start_warning) & (df_temp['Time'] < end_warning)
            if mask.sum() > 0:
                df_temp.loc[mask, 'Anomaly_Warning'] = 1
                df_temp.loc[mask, 'Failure_Type'] = reason_val
                match_count += 1
                
        df_temp['Anomaly_Warning'] = df_temp['Anomaly_Warning'].clip(0, 1)
        print(f"     ✅ Flagged {match_count} downtime risk patterns for {machine_name}.")
        
        processed_dfs.append(df_temp)

    # 4. GỘP DATA ĐÃ CÓ MACHINE_TYPE VÀ WINDOW FEATURES
    df_sensor = pd.concat(processed_dfs, ignore_index=True)
    
    print("\n📊 Target Variable Distribution across ALL machines:")
    display(df_sensor['Anomaly_Warning'].value_counts())
    print("\n🛒 Breakdown by Machine Type:")
    if 'Machine_Type' in df_sensor.columns:
        display(df_sensor.groupby('Machine_Type')['Anomaly_Warning'].value_counts())

    final_path = '/kaggle/working/Theme3_ML_Ready_All_Sensors.csv'
    df_sensor.to_csv(final_path, index=False)
    print(f"\n💾 Saved Machine Learning ready dataset to: {final_path}")

except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"❌ Error: {e}")


## Task 1

In [ ]:
"""
TASK 1: QUALITY PREDICTION & SET POINT OPTIMIZATION (PRESCRIPTIVE ENGINE)

This module implements the complete pipeline for Task 1, upgraded to Machine Learning Standards:
1. Step 1: Align data by shifting to Window-Based Features (trend, stability, change).
2. Step 2: Feature Selection (retain top 30 most important signals to reduce noise).
3. Step 3: Train Specialist Models using LightGBM per 'Part' (Yeast Type).
4. Step 4: Prescriptive Engine using Optuna.
"""

import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("🧠 TASK 1: INITIATING QUALITY PREDICTION AND OPTIMIZATION PIPELINE...\n" + "="*60)

QUALITY_PATH = '/kaggle/working/Theme3_Merged_Quality_Data_FIXED.csv'

try:
    # ---------------------------------------------------------
    # 1. LOAD & PREPARE DATA
    # ---------------------------------------------------------
    df_quality = pd.read_csv(QUALITY_PATH)
    print(f"✅ Loaded Master Quality Dataset: {df_quality.shape[0]} rows, {df_quality.shape[1]} columns")
    
    # Standardize time handling for continuous metrics
    df_quality['Set Time'] = pd.to_datetime(df_quality['Set Time'], errors='coerce')
    df_quality = df_quality.dropna(subset=['Set Time']).sort_values('Set Time').reset_index(drop=True)
    
    # ---------------------------------------------------------
    # 2. WINDOW FEATURE ENGINEERING (CRITICAL REFACTOR)
    # ---------------------------------------------------------
    print("⚙️ Engineering WINDOW-BASED temporal features (mean, std, min, max, delta)...")
    df_quality['Hour'] = df_quality['Set Time'].dt.hour
    
    numeric_cols = df_quality.select_dtypes(include='number').columns
    sp_cols = [c for c in numeric_cols if 'SP' in c.upper()]
    pv_cols = [c for c in numeric_cols if 'PV' in c.upper()]
    core_sensors = sp_cols + pv_cols
    
    print(f"  -> Processing {len(core_sensors)} core sensors...")
    
    # FIX: Increase window size for industrial systems
    window = 15 # Representing context over 15 time steps
    for col in core_sensors:
        df_quality[f'{col}_mean'] = df_quality[col].rolling(window=window, min_periods=1).mean()
        df_quality[f'{col}_std'] = df_quality[col].rolling(window=window, min_periods=1).std().fillna(0)
        df_quality[f'{col}_max'] = df_quality[col].rolling(window=window, min_periods=1).max()
        df_quality[f'{col}_min'] = df_quality[col].rolling(window=window, min_periods=1).min()
        # delta = current - value steps ago to capture the rate of change (slope)
        df_quality[f'{col}_delta']  = df_quality[col] - df_quality[col].shift(window).fillna(method='bfill')
    
    # Time-series safe missing value imputation: Prevent Data Leakage
    numeric_features = df_quality.select_dtypes(include='number').columns
    df_quality[numeric_features] = df_quality[numeric_features].ffill().fillna(0)
    
    # Fill non-numeric sequentially
    df_quality = df_quality.ffill()
    
    # ---------------------------------------------------------
    # 3. LABEL ENCODING & TARGET SHIFTING
    # ---------------------------------------------------------
    label_map = {'Good': 0, 'Low_Bad': 1, 'High_Bad': 2}
    
    if 'Quality_Label' in df_quality.columns:
        df_quality['Target'] = df_quality['Quality_Label'].map(label_map)
        
        # 🔥 FIX: Shift target to predict future quality (Prediction Horizon) 
        # Aligned with window latency so prediction isn't too narrow
        prediction_horizon = 15 
        print(f"🔮 Shifting target {prediction_horizon} steps into the future for true prediction capability.")
        df_quality['Target'] = df_quality['Target'].shift(-prediction_horizon)
        
        df_quality = df_quality.dropna(subset=['Target'])
        
        # Prepare feature space
        exclude_cols = ['Target', 'Quality_Label', 'Set Time', 'VYP batch', 'Part', 'Class']
        initial_feature_cols = [c for c in df_quality.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df_quality[c])]
        
        models_per_part = {}
        parts = df_quality['Part'].unique()
        low_bad_idx = 1
        
        # ---------------------------------------------------------
        # 4. TRAINING SPECIALIST MODELS (LIGHTGBM) ALONG WITH FEATURE SELECTION
        # ---------------------------------------------------------
        for part in parts:
            df_part = df_quality[df_quality['Part'] == part]
            if len(df_part) < 50:
                print(f"\n⚠️ Part [{part}]: Insufficient data ({len(df_part)} rows). Skipping.")
                continue
                
            print(f"\n🚀 Processing Part: {part} | Profile: {len(df_part)} records")
            
            # Clean feature names to avoid JSON character errors in LightGBM
            clean_feature_cols = [re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in initial_feature_cols]
            df_part_renamed = df_part.rename(columns=dict(zip(initial_feature_cols, clean_feature_cols)))
            
            X = df_part_renamed[clean_feature_cols]
            y = df_part_renamed['Target']
            
            # temporal split
            split_idx = int(len(X) * 0.8)
            X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
            y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
            
            # -- QUICK FEATURE SELECTION (Top 15 SP + Top 15 PV) - BEFORE SMOTE FIX --
            print(f"  -> Performing Feature Selection to reduce noise (Top 15 SP + Top 15 PV)...")
            selector_model = lgb.LGBMClassifier(n_estimators=50, random_state=42, verbose=-1, class_weight='balanced')
            selector_model.fit(X_train, y_train)
            
            feature_imp = pd.Series(selector_model.feature_importances_, index=clean_feature_cols)
            
            # Split features into SP and PV
            sp_feats = [f for f in clean_feature_cols if 'SP' in f.upper()]
            pv_feats = [f for f in clean_feature_cols if f not in sp_feats]
            
            top_sp = feature_imp[sp_feats].nlargest(15).index.tolist() if sp_feats else []
            top_pv = feature_imp[pv_feats].nlargest(15).index.tolist() if pv_feats else []
            
            top_30_features = top_sp + top_pv
            
            # Trim datasets to top selected signal-rich features
            X_train = X_train[top_30_features]
            X_test = X_test[top_30_features]
            
            # SMOTE for imbalance AFTER feature selection
            try:
                sm_k = min(3, y_train.value_counts().min() - 1)
                if sm_k > 0:
                    smote = SMOTE(k_neighbors=sm_k, random_state=42)
                    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
                else:
                    X_train_res, y_train_res = X_train, y_train
            except:
                X_train_res, y_train_res = X_train, y_train
                
            # -- TRAIN MAIN LIGHTGBM MODEL --
            base_model = lgb.LGBMClassifier(
                n_estimators=150, 
                max_depth=6, 
                learning_rate=0.05, 
                random_state=42, 
                verbose=-1,
                class_weight='balanced'
            )
            base_model.fit(X_train_res, y_train_res)
            
            # 🔥 PROBABILITY CALIBRATION (Rất quan trọng cho Optuna)
            try:
                print("  -> Calibrating Probabilities for High Confidence Prescriptions...")
                # Calibrate on Un-SMOTEd Validation Data for Real-World Likelihood
                model = CalibratedClassifierCV(estimator=base_model, method='isotonic', cv='prefit')
                model.fit(X_train, y_train)
            except Exception as calib_e:
                print(f"  -> Calibration failed: {calib_e}. Using raw model.")
                model = base_model
                
            models_per_part[part] = {'model': model, 'features': top_30_features}
            
            # Evaluation
            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
            print(f"  -> Metrics | Accuracy: {acc:.4f} | F1-Score: {f1:.4f}")
            
            # Visualization 1: Feature Importance
            import matplotlib.pyplot as plt
            import seaborn as sns
            
            plt.figure(figsize=(10, 5))
            feature_imp_final = pd.Series(base_model.feature_importances_, index=top_30_features).sort_values(ascending=False)
            sns.barplot(x=feature_imp_final.head(15).values, y=feature_imp_final.head(15).index, palette='viridis')
            plt.title(f"Top 15 Predictive Features for Quality Prediction - Part: {part}")
            plt.xlabel("LightGBM Importance Score")
            plt.ylabel("Features")
            plt.tight_layout()
            plt.show()
            
        # ---------------------------------------------------------
        # 5. BAYESIAN OPTIMIZATION: PRESCRIPTIVE ENGINE
        # ---------------------------------------------------------
        print("\n" + "="*60)
        print("🎯 PRESCRIPTIVE ENGINE: BAYESIAN OPTIMIZATION (OPTUNA)")
        
        if models_per_part and len(parts) > 0:
            sample_part = list(models_per_part.keys())[0]
            sample_model_dict = models_per_part[sample_part]
            sample_model = sample_model_dict['model']
            selected_feats = sample_model_dict['features']
            
            print(f"Generating optimized Set Points for '{sample_part}'...")
            
            # Snapshot the current production state
            clean_feature_cols_map = {col: re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in initial_feature_cols}
            df_quality_renamed = df_quality.rename(columns=clean_feature_cols_map)
            
            # get recent historical window for feature recomputation
            # FIX: We need window + 1 rows to accurately compute a pandas shift(window) delta
            recent_window_sz = window + 1
            df_historical = df_quality_renamed[df_quality_renamed['Part'] == sample_part].iloc[-recent_window_sz:].copy()
            df_current_state = df_historical.iloc[-1].copy()
            
            # Identify core SP columns to optimize that are in our model or derived features
            sp_to_optimize = []
            for sp in sp_cols:
                clean_sp = clean_feature_cols_map.get(sp, sp)
                # check if the SP or its variants are in selected_feats
                if any(clean_sp in f for f in selected_feats):
                    sp_to_optimize.append(clean_sp)
            
            # Optimize top 15 Set points
            sp_to_optimize = sp_to_optimize[:15]
            
            def sp_optimization_objective(trial):
                # We need to simulate the rolling features with the proposed SP values
                df_sim = df_historical.copy()
                
                # Propose new SP values
                proposed_sps = {}
                penalty_score = 0
                for clean_sp in sp_to_optimize:
                    curr_val = df_current_state[clean_sp] + 1e-9 # avoid division by 0
                    if curr_val > 0:
                        new_val = trial.suggest_float(clean_sp, curr_val * 0.95, curr_val * 1.05)
                    elif curr_val < 0:
                        new_val = trial.suggest_float(clean_sp, curr_val * 1.05, curr_val * 0.95)
                    else:
                        new_val = trial.suggest_float(clean_sp, -0.05, 0.05)
                        
                    proposed_sps[clean_sp] = new_val
                    # 🔥 ADD CONSTRAINT: Penalty for making extreme SP changes
                    penalty_score += abs(new_val - curr_val) / abs(curr_val)
                    
                # Update the last row with proposed SPs
                for sp, val in proposed_sps.items():
                    df_sim.iloc[-1, df_sim.columns.get_loc(sp)] = val
                    
                # 🔥 FIX: Specifically recompute ONLY the features required by selected_feats
                X_simulated_row = pd.DataFrame(index=[0])
                
                for sf in selected_feats:
                    # Identify the base column and calculate the specific statistical feature
                    if sf.endswith('_mean'):
                        base_col = sf[:-5] # remove '_mean'
                        if base_col in df_sim.columns:
                            X_simulated_row[sf] = df_sim[base_col].iloc[-window:].mean()
                        else:
                            X_simulated_row[sf] = df_current_state.get(sf, 0)
                    elif sf.endswith('_std'):
                        base_col = sf[:-4] # remove '_std'
                        if base_col in df_sim.columns:
                            X_simulated_row[sf] = df_sim[base_col].iloc[-window:].std()
                        else:
                            X_simulated_row[sf] = df_current_state.get(sf, 0)
                    elif sf.endswith('_max'):
                        base_col = sf[:-4] # remove '_max'
                        if base_col in df_sim.columns:
                            X_simulated_row[sf] = df_sim[base_col].iloc[-window:].max()
                        else:
                            X_simulated_row[sf] = df_current_state.get(sf, 0)
                    elif sf.endswith('_min'):
                        base_col = sf[:-4] # remove '_min'
                        if base_col in df_sim.columns:
                            X_simulated_row[sf] = df_sim[base_col].iloc[-window:].min()
                        else:
                            X_simulated_row[sf] = df_current_state.get(sf, 0)
                    elif sf.endswith('_delta'):
                        base_col = sf[:-6] # remove '_delta'
                        if base_col in df_sim.columns:
                            # EXACT MATCH WITH TRAINING: df[col] - df[col].shift(window)
                            X_simulated_row[sf] = df_sim[base_col].iloc[-1] - df_sim[base_col].iloc[-(window + 1)]
                        else:
                            X_simulated_row[sf] = df_current_state.get(sf, 0)
                    else:
                        # Direct base column (no rolling suffix)
                        if sf in df_sim.columns:
                            X_simulated_row[sf] = df_sim[sf].iloc[-1]
                        else:
                            X_simulated_row[sf] = df_current_state.get(sf, 0)
                            
                # Keep exact order
                X_sim_final = X_simulated_row[selected_feats]
                
                probabilities = sample_model.predict_proba(X_sim_final)
                prob_good = probabilities[0][0] # Assuming class 0 is Good
                
                # Apply penalty for sensible optimization (Lamda * Penalty)
                lambda_penalty = 0.02
                objective_score = prob_good - (lambda_penalty * penalty_score)
                
                return objective_score
            
            optuna.logging.set_verbosity(optuna.logging.WARNING)
            study = optuna.create_study(direction="maximize")
            study.optimize(sp_optimization_objective, n_trials=100)
            
            print(f"\n✅ OPTIMIZATION RESULTS:")
            # Unpack real unpenalized prob if needed, but best_value output includes penalty
            print(f"  -> Evaluated Optimized Metric (Soft-P + Constraints): {study.best_value:.4f}")
            print(f"  -> ✨ Recommended Set Points:")
            
            opt_sps = []
            orig_vals = []
            new_vals = []
            
            for sp_name, optimized_val in study.best_params.items():
                original_val = df_current_state[sp_name]
                change_pct = ((optimized_val - original_val) / (original_val + 1e-9)) * 100
                print(f"      🔹 {sp_name}: {optimized_val:.3f} | Offset: {change_pct:+.1f}% from original {original_val:.3f}")
                
                opt_sps.append(sp_name)
                orig_vals.append(original_val)
                new_vals.append(optimized_val)
                
            # Visualization 2: Optimized SPs vs Original SPs
            import matplotlib.pyplot as plt
            import numpy as np
            
            x = np.arange(len(opt_sps))
            width = 0.35
            
            fig, ax = plt.subplots(figsize=(12, 6))
            ax.bar(x - width/2, orig_vals, width, label='Original SP', color='#ff9999')
            ax.bar(x + width/2, new_vals, width, label='Recommended SP', color='#66b3ff')
            
            ax.set_ylabel('Set Point Value')
            ax.set_title(f'Prescriptive Engine: Set Point Optimization ({sample_part})')
            ax.set_xticks(x)
            ax.set_xticklabels(opt_sps, rotation=45, ha='right')
            ax.legend()
            
            plt.tight_layout()
            plt.show()

except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"❌ Pipeline Execution Failed: {e}")

## Task 2: Downtime

In [ ]:
"""
TASK 2: DOWNTIME DETECTION (ANOMALY DETECTION)

Dữ liệu ML_Ready_All_Sensors.csv bây giờ ĐÃ CÓ SẴN các window features và được phân loại theo Machine_Type.
Model sẽ học các pattern ổn định (mean, std, delta) thay vì nhiễu raw.
"""

import pandas as pd
import numpy as np
import lightgbm as lgb
import re
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, roc_auc_score
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("🚨 TASK 2: INITIATING DOWNTIME RISK CLASSIFICATION...\n" + "="*60)

ML_READY_DATA_PATH = '/kaggle/working/Theme3_ML_Ready_All_Sensors.csv'

try:
    df_dt_ml = pd.read_csv(ML_READY_DATA_PATH)
    df_dt_ml['Time'] = pd.to_datetime(df_dt_ml['Time'])
    df_dt_ml = df_dt_ml.sort_values('Time').reset_index(drop=True)
    
    # FIX: Multi-class labeling - Incorporating both Machine and Failure Type
    df_dt_ml['Failure_Class_Name'] = 'Normal'
    mask = df_dt_ml['Anomaly_Warning'] == 1
    
    if 'Failure_Type' in df_dt_ml.columns and df_dt_ml['Failure_Type'].nunique() > 1:
        # Prevent appending "None" or irrelevant categories
        valid_reason_mask = mask & (df_dt_ml['Failure_Type'] != 'None')
        if 'Machine_Type' in df_dt_ml.columns:
            df_dt_ml.loc[valid_reason_mask, 'Failure_Class_Name'] = df_dt_ml.loc[valid_reason_mask, 'Machine_Type'].astype(str) + "_" + df_dt_ml.loc[valid_reason_mask, 'Failure_Type'].astype(str)
        else:
            df_dt_ml.loc[valid_reason_mask, 'Failure_Class_Name'] = df_dt_ml.loc[valid_reason_mask, 'Failure_Type'].astype(str)
            
        # Fallback for remaining mask rows without specific failure type
        residual_mask = mask & (df_dt_ml['Failure_Class_Name'] == 'Normal')
        if 'Machine_Type' in df_dt_ml.columns:
            df_dt_ml.loc[residual_mask, 'Failure_Class_Name'] = df_dt_ml.loc[residual_mask, 'Machine_Type'].astype(str) + "_Unknown_Failure"
        else:
            df_dt_ml.loc[residual_mask, 'Failure_Class_Name'] = 'Generic_Failure'
    else:
        if 'Machine_Type' in df_dt_ml.columns:
            df_dt_ml.loc[mask, 'Failure_Class_Name'] = df_dt_ml.loc[mask, 'Machine_Type'].astype(str) + "_Failure"
        else:
            df_dt_ml.loc[mask, 'Failure_Class_Name'] = 'Generic_Failure'

    # Sanitize class names for LightGBM
    df_dt_ml['Failure_Class_Name'] = df_dt_ml['Failure_Class_Name'].apply(lambda x: re.sub(r'[^A-Za-z0-9_]+', '_', str(x)))

    # Label Encoding for LightGBM ensuring 'Normal' is always mapped to 0
    unique_classes = df_dt_ml['Failure_Class_Name'].unique()
    class_map = {name: idx for idx, name in enumerate(unique_classes)}
    if 'Normal' in class_map and class_map['Normal'] != 0:
        normal_idx = class_map['Normal']
        for k, v in class_map.items():
            if v == 0:
                class_map[k] = normal_idx
        class_map['Normal'] = 0

    df_dt_ml['Failure_Class'] = df_dt_ml['Failure_Class_Name'].map(class_map)
    idx_to_class = {v: k for k, v in class_map.items()}

    # Print overall distribution
    print("Multi-class detailed target distribution:")
    print(df_dt_ml['Failure_Class_Name'].value_counts())
    
    # We train a unified model to predict WHICH machine will fail AND WHAT the failure is
    print(f"\n🚀 TRAINING UNIFIED MULTI-CLASS MODEL FOR ROOT CAUSE ROOT ANALYSIS")
        
    exclude_from_features = ['Time', 'Anomaly_Warning', 'Failure_Class', 'Failure_Class_Name', 'Failure_Type', 'Set Time', 'VYP batch', 'Part', 'Machine_Type']
    features = [c for c in df_dt_ml.select_dtypes(include='number').columns if c not in exclude_from_features]
    
    # Điền khuyết an toàn cho Time-Series: Prevent Data Leakage
    df_dt_ml[features] = df_dt_ml[features].ffill().fillna(0)
    
    # Format the features names to remove special characters (JSON/LightGBM compatibility)
    clean_features = [re.sub(r'[^A-Za-z0-9_]+', '_', f) for f in features]
    df_dt_ml = df_dt_ml.rename(columns=dict(zip(features, clean_features)))
    
    X = df_dt_ml[clean_features]
    y = df_dt_ml['Failure_Class']
    
    print(f"  -> Ingesting {len(X)} rows with {len(clean_features)} Window-Based features.")
    
    # ---------------------------------------------------------
    # CHRONOLOGICAL TRAIN/TEST SPLIT
    # ---------------------------------------------------------
    split_idx = int(len(df_dt_ml) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    
    # -- QUICK FEATURE SELECTION (Top 30 signals to reduce noise) --
    print(f"  -> Selecting Top 30 Predictive Features...")
    # Use objective='multiclass' for the selector
    lgb_selector = lgb.LGBMClassifier(n_estimators=50, random_state=42, verbose=-1, class_weight='balanced', objective='multiclass')
    lgb_selector.fit(X_train, y_train)
    
    feature_imp_dt = pd.Series(lgb_selector.feature_importances_, index=clean_features)
    top_30_features_dt = feature_imp_dt.nlargest(30).index.tolist()
    
    X_train = X_train[top_30_features_dt]
    X_test = X_test[top_30_features_dt]
    
    # 🔥 Multi-Class SMOTE (Downtime class imbalance handling)
    print("  -> Applying SMOTE to balance multi-class downtime events safely...")
    try:
        min_class_count = y_train.value_counts().min()
        sm_k = min(3, min_class_count - 1)
        if sm_k > 0:
            smote = SMOTE(k_neighbors=sm_k, random_state=42)
            X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
        else:
            X_train_res, y_train_res = X_train, y_train
    except Exception as smote_e:
        print(f"  -> SMOTE fallback: {smote_e}. Using original imbalanced dataset.")
        X_train_res, y_train_res = X_train, y_train
    
    # =====================================================================
    # MODEL 1: LIGHTGBM (SUPERVISED MULTICLASS)
    # =====================================================================
    print("  -> Training LightGBM Multi-class Classifier...")
    lgb_dt = lgb.LGBMClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        class_weight='balanced',
        objective='multiclass',
        random_state=42,
        verbose=-1
    )
    lgb_dt.fit(X_train_res, y_train_res)
    y_pred_lgb = lgb_dt.predict(X_test)
    
    # =====================================================================
    # MODEL 2: ISOLATION FOREST (FIX: Scaled)
    # =====================================================================
    print("  -> Training Isolation Forest (Normal Behaviors Only)...")
    X_train_normal = X_train[y_train == 0]
    
    # FIX: Scale data for Isolation Forest
    scaler = StandardScaler()
    X_train_normal_scaled = scaler.fit_transform(X_train_normal)
    X_test_scaled = scaler.transform(X_test)
    
    iso_forest = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1)
    iso_forest.fit(X_train_normal_scaled)
    
    # Iso forest predicts -1 for anomaly, 1 for normal
    iso_preds = iso_forest.predict(X_test_scaled)
    iso_scores = iso_forest.decision_function(X_test_scaled)
    # Convert to binary anomaly indicator
    y_pred_iso_anomaly = np.where(iso_preds == -1, 1, 0)
    # Create binary true labels for evaluation against IsoForest
    y_test_binary = (y_test > 0).astype(int)
    
    # =====================================================================
    # METRICS SUMMARY & SHAP-LIKE INTERPRETATION
    # =====================================================================
    print("📊 RESULTS:")

    # Classification Report dynamically mapping index to class names
    y_test_names = [idx_to_class[idx] for idx in y_test]
    y_pred_names = [idx_to_class[idx] for idx in y_pred_lgb]
    print("  | LightGBM Classification Report (Detailed Failure Class):")
    print(classification_report(y_test_names, y_pred_names, zero_division=0))
    
    iso_f1 = f1_score(y_test_binary, y_pred_iso_anomaly, zero_division=0)
    iso_rec = recall_score(y_test_binary, y_pred_iso_anomaly, zero_division=0)
    print(f"  | Iso-For (Binary Anomaly) F1: {iso_f1:.4f} | Recall: {iso_rec:.4f}")
    print(f"  | Iso-For Mean Anomaly Score (Lower is more anomalous): {iso_scores.mean():.4f}")

    # Visualization: Anomaly Score Curve
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12, 5))
    plt.plot(iso_scores, label='Anomaly Score (Decision Function)', color='blue', alpha=0.7)
    plt.axhline(0, color='red', linestyle='--', label='Anomaly Threshold (0)')
    
    # Mark real downtime events
    anomaly_indices = np.where(y_test_binary == 1)[0]
    plt.scatter(anomaly_indices, iso_scores[anomaly_indices], color='red', label='Actual Downtime', zorder=5)
    
    plt.title("Isolation Forest: Anomaly Score Curve over Time")
    plt.xlabel("Test Data Time Steps")
    plt.ylabel("Anomaly Score (Lower = More Anomalous)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("📊 RESULTS:" + "="*60)
    print("📈 DOWNTIME ROOT CAUSE ANALYSIS (TOP 5 VITAL SENSORS)")
    final_imp = pd.Series(lgb_dt.feature_importances_, index=top_30_features_dt).sort_values(ascending=False)
    for feat, imp in final_imp.head(5).items():
        print(f"  🔹 {feat}: {imp:.2f}")
        
    print("\n🚨 REAL-TIME RISK ALERT SIMULATION (LATEST WINDOW):")
    recent_sample = X_test.iloc[-1:]
    pred_class_idx = lgb_dt.predict(recent_sample)[0]
    pred_prob = lgb_dt.predict_proba(recent_sample)[0]
    
    pred_class_name = idx_to_class[pred_class_idx]
    
    if pred_class_idx == 0:
        print(f"  ✅ System Status: NORMAL (Confidence: {pred_prob[0]:.2%})")
    else:
        risk_prob = pred_prob[pred_class_idx]
        print(f"  ⚠️ DOWNTIME RISK DETECTED!")
        
        # Parse machine vs root cause (e.g. TFE_overheat)
        parts = pred_class_name.split('_', 1)
        machine = parts[0] if len(parts) > 0 else "Unknown"
        failure = parts[1] if len(parts) > 1 else pred_class_name
        
        print(f"  → Target Machine: {machine}")
        print(f"  → Predicted Failure Cause: {failure}")
        print(f"  → Risk Level: HIGH (Probability: {risk_prob:.2%})")
        print(f"  → Key driving factor building instability: {final_imp.index[0]}")

except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"❌ Error during Task 2 Execution: {e}")


## Export Models & Configs for Backend

In [ ]:
import joblib
import json
import os

print("📦 INITIATING MODEL & CONFIG EXPORT FOR BACKEND...\n" + "="*60)

# Create folders if they don't exist
os.makedirs('/kaggle/working/models', exist_ok=True)
os.makedirs('/kaggle/working/config', exist_ok=True)

# === EXPORT TASK 1 ===
try:
    task1_features_dict = {}
    for part, model_info in models_per_part.items():
        safe_part_name = part.replace(" ", "_").replace("-", "")
        # Dump model
        joblib.dump(model_info['model'], f'/kaggle/working/models/task1_model_{safe_part_name}.joblib')
        print(f"  ✅ Exported Task 1 Model: task1_model_{safe_part_name}.joblib")
        # Save feature list for the config
        task1_features_dict[safe_part_name] = model_info['features']
        
    # Dump all Task 1 features into a single config file
    with open('/kaggle/working/config/task1_features.json', 'w') as f:
        json.dump(task1_features_dict, f)
    print("  ✅ Exported Task 1 Features Config: task1_features.json")

except Exception as e:
    print(f"  ❌ Error exporting Task 1 models: {e}")

# === EXPORT TASK 2 ===
try:
    joblib.dump(lgb_dt, '/kaggle/working/models/task2_lightgbm_multiclass.joblib')
    print("  ✅ Exported Task 2 LightGBM Model: task2_lightgbm_multiclass.joblib")
    
    joblib.dump(iso_forest, '/kaggle/working/models/task2_isolation_forest.joblib')
    print("  ✅ Exported Task 2 Isolation Forest Model: task2_isolation_forest.joblib")
    
    # Rất quan trọng! Scale dữ liệu trước khi đưa vào mô hình là bước sống còn.
    joblib.dump(scaler, '/kaggle/working/models/task2_scaler.joblib') 
    print("  ✅ Exported Task 2 Scaler: task2_scaler.joblib")

    with open('/kaggle/working/config/task2_features.json', 'w') as f:
        json.dump(top_30_features_dt, f)
    print("  ✅ Exported Task 2 Features Config: task2_features.json")

    with open('/kaggle/working/config/task2_class_mapping.json', 'w') as f:
        json.dump(idx_to_class, f)
    print("  ✅ Exported Task 2 Class Mapping Config: task2_class_mapping.json")

except Exception as e:
    print(f"  ❌ Error exporting Task 2 models: {e}")

print("\n🎉 Đã xuất kho thành công toàn bộ Model và Config cho Backend!")